In [2]:
#importing the libraries 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

In [2]:
data=pd.read_csv("/kaggle/input/ipl-dataset-2008-to-2025/ball_by_ball_data.csv")
print(data)

        season_id  match_id        batter          bowler   non_striker  \
0            2008    335982    SC Ganguly         P Kumar   BB McCullum   
1            2008    335982   BB McCullum         P Kumar    SC Ganguly   
2            2008    335982   BB McCullum         P Kumar    SC Ganguly   
3            2008    335982   BB McCullum         P Kumar    SC Ganguly   
4            2008    335982   BB McCullum         P Kumar    SC Ganguly   
...           ...       ...           ...             ...           ...   
278200       2025   1485779      T Stubbs  Arshdeep Singh  Sameer Rizvi   
278201       2025   1485779  Sameer Rizvi      MP Stoinis      T Stubbs   
278202       2025   1485779  Sameer Rizvi      MP Stoinis      T Stubbs   
278203       2025   1485779      T Stubbs      MP Stoinis  Sameer Rizvi   
278204       2025   1485779  Sameer Rizvi      MP Stoinis      T Stubbs   

        team_batting  team_bowling  over_number  ball_number  batter_runs  \
0                  6  

In [3]:
data.isnull().sum()

season_id                 0
match_id                  0
batter                    0
bowler                    0
non_striker               0
team_batting              0
team_bowling              0
over_number               0
ball_number               0
batter_runs               0
extras                    0
total_runs                0
batsman_type              0
bowler_type               0
player_out           264382
fielders_involved    264382
is_wicket                 0
is_wide_ball              0
is_no_ball                0
is_leg_bye                0
is_bye                    0
is_penalty                0
wide_ball_runs            0
no_ball_runs              0
leg_bye_runs              0
bye_runs                  0
penalty_runs              0
wicket_kind          264382
is_super_over             0
innings                   0
dtype: int64

In [9]:
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = data.select_dtypes(include=['object']).columns


In [10]:
data[num_cols] = data[num_cols].fillna(data[num_cols].mean())


In [11]:
for col in cat_cols:
    data[col].fillna(data[col].mode()[0], inplace=True)


/tmp/ipykernel_47/2674119437.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].mode()[0], inplace=True)


In [12]:
print(data.isnull().sum())


season_id            0
match_id             0
batter               0
bowler               0
non_striker          0
team_batting         0
team_bowling         0
over_number          0
ball_number          0
batter_runs          0
extras               0
total_runs           0
batsman_type         0
bowler_type          0
player_out           0
fielders_involved    0
is_wicket            0
is_wide_ball         0
is_no_ball           0
is_leg_bye           0
is_bye               0
is_penalty           0
wide_ball_runs       0
no_ball_runs         0
leg_bye_runs         0
bye_runs             0
penalty_runs         0
wicket_kind          0
is_super_over        0
innings              0
dtype: int64


In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278205 entries, 0 to 278204
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   season_id          278205 non-null  int64 
 1   match_id           278205 non-null  int64 
 2   batter             278205 non-null  object
 3   bowler             278205 non-null  object
 4   non_striker        278205 non-null  object
 5   team_batting       278205 non-null  int64 
 6   team_bowling       278205 non-null  int64 
 7   over_number        278205 non-null  int64 
 8   ball_number        278205 non-null  int64 
 9   batter_runs        278205 non-null  int64 
 10  extras             278205 non-null  int64 
 11  total_runs         278205 non-null  int64 
 12  batsman_type       278205 non-null  object
 13  bowler_type        278205 non-null  object
 14  player_out         278205 non-null  object
 15  fielders_involved  278205 non-null  object
 16  is_wicket          2

In [41]:
batsman_match = (
    data.groupby(['match_id', 'batter', 'team_batting', 'innings'])
        .agg(
            runs=('batter_runs', 'sum'),
            balls=('ball_number', 'count'),
            fours=('batter_runs', lambda x: (x == 4).sum()),
            sixes=('batter_runs', lambda x: (x == 6).sum()),
            outs=('is_wicket', 'sum')
        )
        .reset_index()
)


In [42]:
batsman_match['strike_rate'] = (
    batsman_match['runs'] / batsman_match['balls']
) * 100


In [43]:
batsman_match = batsman_match.sort_values(
    by=['batter', 'match_id']
).reset_index(drop=True)


In [44]:
batsman_match['runs_last_5'] = (
    batsman_match.groupby('batter')['runs']
    .shift(1)
    .rolling(5)
    .mean()
)


In [45]:
batsman_match['career_runs'] = (
    batsman_match.groupby('batter')['runs']
    .cumsum()
    - batsman_match['runs']
)


In [46]:
batsman_match['career_matches'] = (
    batsman_match.groupby('batter').cumcount()
)


In [47]:
batsman_match['career_avg'] = (
    batsman_match['career_runs'] /
    batsman_match['career_matches'].replace(0, np.nan)
)


In [49]:
batsman_match['runs_last_10'] = (
    batsman_match.groupby('batter')['runs']
    .shift(1)
    .rolling(10)
    .mean()
)


In [50]:
batsman_match.fillna({
    'runs_last_5': batsman_match['runs_last_5'].mean(),
    'runs_last_10': batsman_match['runs_last_10'].mean(),
    'career_avg': batsman_match['career_avg'].mean()
}, inplace=True)


In [51]:
batsman_match['career_runs'] = (
    batsman_match.groupby('batter')['runs']
    .cumsum()
    - batsman_match['runs']
)


In [52]:
batsman_match['career_matches'] = (
    batsman_match.groupby('batter').cumcount()
)


In [53]:
batsman_match['career_avg'] = (
    batsman_match['career_runs'] /
    batsman_match['career_matches'].replace(0, np.nan)
)


In [54]:
batsman_match.fillna({
    'runs_last_5': batsman_match['runs_last_5'].mean(),
    'runs_last_10': batsman_match['runs_last_10'].mean(),
    'career_avg': batsman_match['career_avg'].mean()
}, inplace=True)


In [55]:
cat_features = ['batter', 'team_batting', 'innings']

num_features = [
    'runs_last_5',
    'runs_last_10',
    'career_avg',
    'strike_rate'
]


In [56]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', MinMaxScaler(), num_features)
    ]
)


In [57]:
X = batsman_match[cat_features + num_features]
y = batsman_match['runs']


In [58]:
split_index = int(0.8 * len(batsman_match))

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]


In [59]:
batsman_match.to_csv("dataset.csv", index=False)


In [62]:
print(batsman_match.head())
print(batsman_match.shape)


   match_id          batter  team_batting  innings  runs  balls  fours  sixes  \
0    548346  A Ashish Reddy             2        1    10     10      0      1   
1    548352  A Ashish Reddy             2        2     3      3      0      0   
2    548359  A Ashish Reddy             2        2     8      8      1      0   
3    548373  A Ashish Reddy             2        2    10      4      2      0   
4    548376  A Ashish Reddy             2        1     4      5      0      0   

   outs  strike_rate  runs_last_5  career_runs  career_matches  career_avg  \
0     1        100.0     21.54637            0               0   20.192986   
1     1        100.0     21.54637           10               1   10.000000   
2     1        100.0     21.54637           13               2    6.500000   
3     0        250.0     21.54637           21               3    7.000000   
4     1         80.0     21.54637           31               4    7.750000   

   runs_last_10  
0     22.368528  
1     22

In [63]:
batsman_match.head()


,match_id,batter,team_batting,innings,runs,balls,fours,sixes,outs,strike_rate,runs_last_5,career_runs,career_matches,career_avg,runs_last_10
0,548346,A Ashish Reddy,2,1,10,10,0,1,1,100.0,21.54637,0,0,20.192986,22.368528
1,548352,A Ashish Reddy,2,2,3,3,0,0,1,100.0,21.54637,10,1,10.000000,22.368528
2,548359,A Ashish Reddy,2,2,8,8,1,0,1,100.0,21.54637,13,2,6.500000,22.368528
3,548373,A Ashish Reddy,2,2,10,4,2,0,0,250.0,21.54637,21,3,7.000000,22.368528
4,548376,A Ashish Reddy,2,1,4,5,0,0,1,80.0,21.54637,31,4,7.750000,22.368528


In [64]:
batsman_match.shape


(17708, 15)

In [65]:
batsman_match[['runs', 'runs_last_5', 'career_avg']].head()


,runs,runs_last_5,career_avg
0,10,21.54637,20.192986
1,3,21.54637,10.000000
2,8,21.54637,6.500000
3,10,21.54637,7.000000
4,4,21.54637,7.750000
